In [1]:
import requests
import pandas as pd
from tabulate import tabulate
from datetime import datetime, timezone
import argparse
import sys

In [2]:
BASE_URL = "https://api.lyra.finance"

In [22]:
def api_post(endpoint: str, params: dict) -> dict:
    """POST to Derive REST API and return the result."""
    url = f"{BASE_URL}{endpoint}"
    resp = requests.post(url, json=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    if "error" in data:
        raise RuntimeError(f"API error: {data['error']}")
    return data["result"]

In [4]:
def get_instruments(currency: str) -> list[dict]:
    """Fetch all active option instruments for a currency."""
    return api_post("/public/get_instruments", {
        "currency": currency,
        "instrument_type": "option",
        "expired": False,
    })

In [5]:
def get_tickers(currency: str, expiry_date: str) -> dict:
    """
    Fetch ticker data (bid/ask/greeks/IV/OI) for all options
    of a given currency and expiry.
    expiry_date format: 'YYYYMMDD'
    """
    result = api_post("/public/get_tickers", {
        "instrument_type": "option",
        "currency": currency,
        "expiry_date": expiry_date,
    })
    return result.get("tickers", {})

In [6]:
def get_spot_price(currency: str) -> float:
    """Get current spot price for a currency."""
    result = api_post("/public/get_currency", {"currency": currency})
    return float(result.get("spot_price", 0))

In [7]:
def parse_expiries(instruments: list[dict]) -> list[str]:
    """Extract sorted unique expiry dates (YYYYMMDD) from instruments."""
    expiries = set()
    for inst in instruments:
        details = inst.get("option_details")
        if details:
            ts = details["expiry"]
            dt = datetime.fromtimestamp(ts, tz=timezone.utc)
            expiries.add(dt.strftime("%Y%m%d"))
    return sorted(expiries)

In [10]:
def build_chain(instruments: list[dict], tickers: dict, expiry_date: str) -> pd.DataFrame:
    """
    Build an option chain DataFrame for a given expiry.
    Structure mirrors the Derive website:
      CALLS  |  Strike  |  PUTS
    """
    # Filter instruments for this expiry
    exp_ts_target = datetime.strptime(expiry_date, "%Y%m%d").replace(tzinfo=timezone.utc).timestamp()
 
    rows = {}  # strike -> {call: ..., put: ...}
 
    for inst in instruments:
        details = inst.get("option_details")
        if not details:
            continue
        if abs(details["expiry"] - exp_ts_target) > 86400:  # wrong expiry
            continue
 
        name = inst["instrument_name"]
        strike = float(details["strike"])
        opt_type = details["option_type"]  # 'C' or 'P'
 
        ticker = tickers.get(f"ticker.{name}.raw") or tickers.get(name) or {}
        op = ticker.get("option_pricing") or {}
        stats = ticker.get("stats") or {}
 
        row_data = {
            "bid":    _f(ticker.get("b")),
            "ask":    _f(ticker.get("a")),
            "mark":   _f(ticker.get("M") or op.get("m")),
            "iv":     _pct(op.get("i")),
            "bid_iv": _pct(op.get("bi")),
            "ask_iv": _pct(op.get("ai")),
            "delta":  _f(op.get("d"), 4),
            "gamma":  _f(op.get("g"), 5),
            "theta":  _f(op.get("t"), 4),
            "vega":   _f(op.get("v"), 4),
            "oi":     _f(stats.get("oi"), 1),
            "volume": _f(stats.get("c"), 1),
        }
 
        if strike not in rows:
            rows[strike] = {}
        rows[strike][opt_type] = row_data
 
    if not rows:
        return pd.DataFrame()
 
    # Build flat DataFrame with calls | strike | puts
    records = []
    for strike in sorted(rows.keys()):
        call = rows[strike].get("C", {})
        put  = rows[strike].get("P", {})
 
        records.append({
            # ── CALLS ──
            "C Bid":     call.get("bid",    "-"),
            "C Ask":     call.get("ask",    "-"),
            "C Mark":    call.get("mark",   "-"),
            "C IV":      call.get("iv",     "-"),
            "C Delta":   call.get("delta",  "-"),
            "C OI":      call.get("oi",     "-"),
            # ── STRIKE ──
            "Strike":    f"{strike:,.0f}",
            # ── PUTS ──
            "P Bid":     put.get("bid",     "-"),
            "P Ask":     put.get("ask",     "-"),
            "P Mark":    put.get("mark",    "-"),
            "P IV":      put.get("iv",      "-"),
            "P Delta":   put.get("delta",   "-"),
            "P OI":      put.get("oi",      "-"),
        })
 
    return pd.DataFrame(records)
 
 

In [11]:
def _f(val, decimals: int = 2) -> str:
    """Format a decimal/string value, return '-' if missing."""
    if val is None:
        return "-"
    try:
        v = float(val)
        if v == 0:
            return "-"
        return f"{v:,.{decimals}f}"
    except (ValueError, TypeError):
        return "-"

In [12]:
def _pct(val) -> str:
    """Format as percentage, e.g. 0.843 → '84.3%'"""
    if val is None:
        return "-"
    try:
        v = float(val)
        if v == 0:
            return "-"
        return f"{v * 100:.1f}%"
    except (ValueError, TypeError):
        return "-"

In [13]:
def print_chain(df: pd.DataFrame, currency: str, expiry_date: str, spot: float):
    """Pretty-print the option chain to the terminal."""
    if df.empty:
        print("  (no data for this expiry)")
        return
 
    # Format expiry nicely
    dt = datetime.strptime(expiry_date, "%Y%m%d")
    expiry_str = dt.strftime("%d %b %Y").upper()
 
    sep = "─" * 120
    print(f"\n{sep}")
    print(f"  {currency} OPTIONS  │  Expiry: {expiry_str}  │  Spot: ${spot:,.2f}")
    print(sep)
    print(f"  {'─── CALLS ───':^54}  {'STRIKE':^10}  {'─── PUTS ───':^54}")
    print(sep)
 
    print(tabulate(
        df,
        headers="keys",
        tablefmt="simple",
        showindex=False,
        colalign=(
            "right", "right", "right", "right", "right", "right",  # calls
            "center",                                                # strike
            "right", "right", "right", "right", "right", "right",  # puts
        )
    ))
    print(sep)
    print("  Columns: Bid | Ask | Mark | IV | Delta | OI")
    print()
 

In [14]:
def export_csv(df: pd.DataFrame, currency: str, expiry_date: str):
    """Optionally save to CSV."""
    fname = f"{currency}_options_{expiry_date}.csv"
    df.to_csv(fname, index=False)
    print(f"  ✓ Saved to {fname}")
 

In [16]:
def main():
    parser = argparse.ArgumentParser(description="Derive Option Chain Viewer")
    parser.add_argument("--currency", default="BTC", help="Currency: BTC, ETH, SOL, etc.")
    parser.add_argument("--expiry",   default=None,  help="Expiry date YYYYMMDD (default: all)")
    parser.add_argument("--csv",      action="store_true", help="Export each expiry to CSV")
    args = parser.parse_args()
 
    currency = args.currency.upper()
 
    print(f"\n📡 Fetching {currency} option instruments...")
    try:
        instruments = get_instruments(currency)
    except Exception as e:
        print(f"✗ Failed to fetch instruments: {e}")
        sys.exit(1)
 
    print(f"   Found {len(instruments)} active {currency} option contracts")
 
    expiries = parse_expiries(instruments)
    if not expiries:
        print("✗ No active expiries found.")
        sys.exit(1)
 
    # Filter to requested expiry if provided
    if args.expiry:
        if args.expiry not in expiries:
            print(f"✗ Expiry {args.expiry} not found. Available: {', '.join(expiries)}")
            sys.exit(1)
        expiries = [args.expiry]
 
    print(f"   Available expiries: {', '.join(expiries)}\n")
 
    # Fetch spot price
    try:
        spot = get_spot_price(currency)
    except Exception:
        spot = 0.0
 
    # Fetch and display each expiry
    for expiry_date in expiries:
        print(f"📊 Fetching tickers for {expiry_date}...", end=" ", flush=True)
        try:
            tickers = get_tickers(currency, expiry_date)
            print(f"({len(tickers)} tickers)")
        except Exception as e:
            print(f"\n  ✗ Failed: {e}")
            continue
 
        df = build_chain(instruments, tickers, expiry_date)
        print_chain(df, currency, expiry_date, spot)
 
        if args.csv:
            export_csv(df, currency, expiry_date)
 
 


In [23]:
# ── Set your parameters here ──
args_currency = "BTC"     # change to "ETH", "SOL", etc.
args_expiry   = None      # set to "20250627" for a specific expiry, or None for all
args_csv      = False     # set to True to save CSV files

# ── Run ──
currency = args_currency.upper()

print(f"\n📡 Fetching {currency} option instruments...")
instruments = get_instruments(currency)
print(f"   Found {len(instruments)} active {currency} option contracts")

expiries = parse_expiries(instruments)
if args_expiry:
    expiries = [args_expiry]

print(f"   Available expiries: {', '.join(expiries)}\n")

try:
    spot = get_spot_price(currency)
except:
    spot = 0.0

for expiry_date in expiries:
    print(f"📊 Fetching tickers for {expiry_date}...", end=" ", flush=True)
    tickers = get_tickers(currency, expiry_date)
    print(f"({len(tickers)} tickers)")
    df = build_chain(instruments, tickers, expiry_date)
    print_chain(df, currency, expiry_date, spot)
    if args_csv:
        export_csv(df, currency, expiry_date)


📡 Fetching BTC option instruments...
   Found 786 active BTC option contracts
   Available expiries: 20260609, 20260610, 20260611, 20260612, 20260613, 20260619, 20260626, 20260703, 20260731, 20260828, 20260925, 20261225, 20270326, 20270625

📊 Fetching tickers for 20260609... (38 tickers)

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  BTC OPTIONS  │  Expiry: 09 JUN 2026  │  Spot: $63,302.65
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                      ─── CALLS ───                         STRIKE                         ─── PUTS ───                     
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   C Bid     C Ask    C Mark    C IV    C Delta    C OI   Strike      P Bid     P Ask     P Mark    P IV    P Delta    P OI
--------  --------  --------  ------  